# FinTech Reconciliation Pipeline

In [1]:
import pandas as pd
import json

## Step 1 : Data Loading

In [ ]:
ledger = pd.read_csv('../01_data/raw/ledger.csv', encoding="utf-8-sig")
gateway = pd.read_csv('../01_data/raw/gateway.csv', encoding="utf-8-sig")

In [3]:
ledger.head()

,transaction_id,transaction_date,merchant_id,amount_usd,status,payment_method
0,R001,2026-03-01,M001,1200.0,success,UPI
1,R002,2026-03-01,M002,850.0,success,Card
2,R003,2026-03-02,M001,500.0,success,Wallet
3,R004,2026-03-02,M003,2100.0,success,Card
4,R005,2026-03-03,M004,7200.0,success,Card


In [4]:
gateway.head()

,transaction_id,transaction_date,merchant_id,amount_usd,status,payment_method
0,R001,2026-03-01,M001,1200.0,success,UPI
1,R002,2026-03-01,M002,900.0,success,Card
2,R003,2026-03-02,M001,500.0,success,Wallet
3,R005,2026-03-03,M004,7200.0,failed,Card
4,R006,2026-03-03,M002,950.0,success,UPI


In [5]:
print("Ledger Shape:", ledger.shape)
print("Gateway Shape:", gateway.shape)

Ledger Shape: (10, 6)
Gateway Shape: (9, 6)


## Step 2 : Data Validation Checks

In [6]:
print("Ledger Dataset Info :")
print("=" * 40)
ledger.info()

Ledger Dataset Info :
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   transaction_id    10 non-null     object 
 1   transaction_date  10 non-null     object 
 2   merchant_id       10 non-null     object 
 3   amount_usd        10 non-null     float64
 4   status            10 non-null     object 
 5   payment_method    10 non-null     object 
dtypes: float64(1), object(5)
memory usage: 612.0+ bytes


In [7]:
print("Gateway Dataset Info :")
print("=" * 40)
gateway.info()

Gateway Dataset Info :
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9 entries, 0 to 8
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   transaction_id    9 non-null      object 
 1   transaction_date  9 non-null      object 
 2   merchant_id       9 non-null      object 
 3   amount_usd        9 non-null      float64
 4   status            9 non-null      object 
 5   payment_method    9 non-null      object 
dtypes: float64(1), object(5)
memory usage: 564.0+ bytes


In [8]:
# NULL Value Check

print("Ledger Dataset Null Values :")
print("=" * 40)
print(ledger.isnull().sum())

print("\n")

print("Gateway Dataset Null Values :")
print("=" * 40)
print(gateway.isnull().sum())

Ledger Dataset Null Values :
transaction_id      0
transaction_date    0
merchant_id         0
amount_usd          0
status              0
payment_method      0
dtype: int64


Gateway Dataset Null Values :
transaction_id      0
transaction_date    0
merchant_id         0
amount_usd          0
status              0
payment_method      0
dtype: int64


In [9]:
# Duplicate Record Check

print("Ledger Dataset Duplicate Records :", ledger.duplicated().sum())
print("Gateway Dataset Duplicate Records :", gateway.duplicated().sum())

Ledger Dataset Duplicate Records : 0
Gateway Dataset Duplicate Records : 0


In [10]:
# Standardizing Transaction ID

ledger['transaction_id'] = ledger['transaction_id'].str.strip()
gateway['transaction_id'] = gateway['transaction_id'].str.strip()

# Step 3 : Reconciliation Logic

In [11]:
# Transactions Missing in Gateway

missing_in_gateway = ledger[
    ~ledger["transaction_id"].isin(gateway["transaction_id"])
]

missing_in_gateway.head()

,transaction_id,transaction_date,merchant_id,amount_usd,status,payment_method
3,R004,2026-03-02,M003,2100.0,success,Card
9,R010,2026-03-05,M004,2500.0,success,Wallet


In [12]:
print("Number of Records Missing in Gateway:", len(missing_in_gateway))

Number of Records Missing in Gateway: 2


In [13]:
# Transactions Missing in Ledger

missing_in_ledger = gateway[
    ~gateway["transaction_id"].isin(ledger["transaction_id"])
]

missing_in_ledger.head()

,transaction_id,transaction_date,merchant_id,amount_usd,status,payment_method
8,R011,2026-03-05,M003,1800.0,success,Card


In [14]:
print("Number of Records Missing in Gateway:", len(missing_in_ledger))

Number of Records Missing in Gateway: 1


In [15]:
# Merging gateway and ledger on transaction_id

merged = pd.merge(
    gateway,
    ledger,
    on = "transaction_id",
    how = "inner",
    suffixes = ("_gateway", "_ledger")
)

merged.head()

,transaction_id,transaction_date_gateway,merchant_id_gateway,amount_usd_gateway,status_gateway,payment_method_gateway,transaction_date_ledger,merchant_id_ledger,amount_usd_ledger,status_ledger,payment_method_ledger
0,R001,2026-03-01,M001,1200.0,success,UPI,2026-03-01,M001,1200.0,success,UPI
1,R002,2026-03-01,M002,900.0,success,Card,2026-03-01,M002,850.0,success,Card
2,R003,2026-03-02,M001,500.0,success,Wallet,2026-03-02,M001,500.0,success,Wallet
3,R005,2026-03-03,M004,7200.0,failed,Card,2026-03-03,M004,7200.0,success,Card
4,R006,2026-03-03,M002,950.0,success,UPI,2026-03-03,M002,950.0,success,UPI


In [16]:
merged.columns

Index(['transaction_id', 'transaction_date_gateway', 'merchant_id_gateway',
       'amount_usd_gateway', 'status_gateway', 'payment_method_gateway',
       'transaction_date_ledger', 'merchant_id_ledger', 'amount_usd_ledger',
       'status_ledger', 'payment_method_ledger'],
      dtype='object')

# Step 4 : Amount Mismatch Detection

In [17]:
# Identifing Amount Mismatches

amount_mismatches = merged[
    merged["amount_usd_gateway"] != merged["amount_usd_ledger"]
]

amount_mismatches.head()

,transaction_id,transaction_date_gateway,merchant_id_gateway,amount_usd_gateway,status_gateway,payment_method_gateway,transaction_date_ledger,merchant_id_ledger,amount_usd_ledger,status_ledger,payment_method_ledger
1,R002,2026-03-01,M002,900.0,success,Card,2026-03-01,M002,850.0,success,Card
6,R008,2026-03-04,M001,600.0,success,Card,2026-03-04,M001,640.0,success,Card


In [18]:
print("Amount Mismatches:", len(amount_mismatches))

Amount Mismatches: 2


# Step 5 : Status Mismatch Detection

In [19]:
# Identifing Status Mismatches

status_mismatches = merged[
    merged["status_gateway"] != merged["status_ledger"]
]

status_mismatches.head()

,transaction_id,transaction_date_gateway,merchant_id_gateway,amount_usd_gateway,status_gateway,payment_method_gateway,transaction_date_ledger,merchant_id_ledger,amount_usd_ledger,status_ledger,payment_method_ledger
3,R005,2026-03-03,M004,7200.0,failed,Card,2026-03-03,M004,7200.0,success,Card


In [20]:
print("Status Mismatches:", len(status_mismatches))

Status Mismatches: 1


# Step 6 : Final Reconciliation Report

In [21]:
# Reconciliation Report Creation

reconciliation_report = merged.copy()

# Amount match flag
reconciliation_report["amount_match"] = (
    reconciliation_report["amount_usd_gateway"]
    == reconciliation_report["amount_usd_ledger"]
)

# Status match flag
reconciliation_report["status_match"] = (
    reconciliation_report["status_gateway"]
    == reconciliation_report["status_ledger"]
)

reconciliation_report.head()

,transaction_id,transaction_date_gateway,merchant_id_gateway,amount_usd_gateway,status_gateway,payment_method_gateway,transaction_date_ledger,merchant_id_ledger,amount_usd_ledger,status_ledger,payment_method_ledger,amount_match,status_match
0,R001,2026-03-01,M001,1200.0,success,UPI,2026-03-01,M001,1200.0,success,UPI,True,True
1,R002,2026-03-01,M002,900.0,success,Card,2026-03-01,M002,850.0,success,Card,False,True
2,R003,2026-03-02,M001,500.0,success,Wallet,2026-03-02,M001,500.0,success,Wallet,True,True
3,R005,2026-03-03,M004,7200.0,failed,Card,2026-03-03,M004,7200.0,success,Card,True,False
4,R006,2026-03-03,M002,950.0,success,UPI,2026-03-03,M002,950.0,success,UPI,True,True


# Step 7 : Exporting Generated Outputs

In [ ]:
# Exporting Records Missing in gateway

missing_in_gateway.to_csv(
    "../01_data/processed/missing_in_gateway.csv",
    index=False
)

print("missing_in_gateway.csv exported successfully")

missing_in_gateway.csv exported successfully


In [ ]:
# Exporting Records Missing in ledger

missing_in_ledger.to_csv(
    "../01_data/processed/missing_in_ledger.csv",
    index=False
)

print("missing_in_ledger.csv exported successfully")

missing_in_ledger.csv exported successfully


In [ ]:
# Exporting Amount Mismatch Records

amount_mismatches.to_csv(
    "../01_data/processed/amount_mismatches.csv",
    index=False
)

print("amount_mismatches.csv exported successfully")

amount_mismatches.csv exported successfully


In [ ]:
# Exporting Status Mismatch Records

status_mismatches.to_csv(
    "../01_data/processed/status_mismatches.csv",
    index=False
)

print("status_mismatches.csv exported successfully")

status_mismatches.csv exported successfully


In [ ]:
# Exporting Final Reconciliation Report

reconciliation_report.to_csv(
    "../01_data/processed/reconciliation_report.csv",
    index=False
)

print("reconciliation_report.csv exported successfully")

reconciliation_report.csv exported successfully


# Step 8 : Summary metrics Generation

In [27]:
# Generating Reconciliation Summary Metrics

summary_metrics = {
    "total_ledger_rows": len(ledger),
    "total_gateway_rows": len(gateway),

    "missing_in_gateway_count": len(missing_in_gateway),
    "missing_in_ledger_count": len(missing_in_ledger),

    "amount_mismatch_count": len(amount_mismatches),
    "status_mismatch_count": len(status_mismatches),

    "reconciliation_issue_count": (
        len(amount_mismatches) +
        len(status_mismatches)
    ),

    "ledger_total_amount": float(
        ledger["amount_usd"].sum()
    ),

    "gateway_total_amount": float(
        gateway["amount_usd"].sum()
    ),

    "amount_at_risk": float(
        amount_mismatches["amount_usd_ledger"].sum()
    )
}

summary_metrics

{'total_ledger_rows': 10,
 'total_gateway_rows': 9,
 'missing_in_gateway_count': 2,
 'missing_in_ledger_count': 1,
 'amount_mismatch_count': 2,
 'status_mismatch_count': 1,
 'reconciliation_issue_count': 3,
 'ledger_total_amount': 23340.0,
 'gateway_total_amount': 20550.0,
 'amount_at_risk': 1490.0}

In [28]:
# Exporting Summary Metrics

with open("summary_metrics.json", "w") as file:
    json.dump(summary_metrics, file, indent=4)

print("summary_metrics.json exported successfully")

summary_metrics.json exported successfully


In [29]:
# Displaying Summary Metrics JSON content

with open("summary_metrics.json", "r") as file:
    print(file.read())

{
    "total_ledger_rows": 10,
    "total_gateway_rows": 9,
    "missing_in_gateway_count": 2,
    "missing_in_ledger_count": 1,
    "amount_mismatch_count": 2,
    "status_mismatch_count": 1,
    "reconciliation_issue_count": 3,
    "ledger_total_amount": 23340.0,
    "gateway_total_amount": 20550.0,
    "amount_at_risk": 1490.0
}


# Step 9 : JSON Normalization

In [ ]:
# Loading API Response Sample JSON File

with open("../01_data/raw/api_response_sample.json", "r") as file:
    api_data = json.load(file)

api_data

{'generated_at': '2026-03-07T10:00:00Z',
 'source': 'QuickPay Settlement API',
 'batches': [{'batch_id': 'B001',
   'merchant': {'merchant_id': 'M001',
    'merchant_name': 'Alpha Mart',
    'region': 'APAC'},
   'settlements': [{'settlement_id': 'S001',
     'amount_usd': 1520.5,
     'status': 'settled',
     'processed_at': '2026-03-07T08:10:00Z',
     'bank': {'name': 'Bank A', 'country': 'IN'}},
    {'settlement_id': 'S002',
     'amount_usd': 980.0,
     'status': 'pending',
     'processed_at': '2026-03-07T08:45:00Z',
     'bank': {'name': 'Bank A', 'country': 'IN'}},
    {'settlement_id': 'S003',
     'amount_usd': 640.0,
     'status': 'settled',
     'processed_at': '2026-03-07T09:15:00Z',
     'bank': {'name': 'Bank B', 'country': 'SG'}}]},
  {'batch_id': 'B002',
   'merchant': {'merchant_id': 'M004',
    'merchant_name': 'Delta Travels',
    'region': 'US'},
   'settlements': [{'settlement_id': 'S004',
     'amount_usd': 2100.0,
     'status': 'settled',
     'processed_at'

In [31]:
type(api_data)

dict

In [32]:
# Flattening Nested JSON Structure

api_normalize = pd.json_normalize(
    api_data['batches'],
    record_path = 'settlements',
    meta = [
        'batch_id',
        ['merchant', 'merchant_id'],
        ['merchant', 'merchant_name'],
        ['merchant', 'region']
    ]
)

api_normalize.head()

,settlement_id,amount_usd,status,processed_at,bank.name,bank.country,batch_id,merchant.merchant_id,merchant.merchant_name,merchant.region
0,S001,1520.5,settled,2026-03-07T08:10:00Z,Bank A,IN,B001,M001,Alpha Mart,APAC
1,S002,980.0,pending,2026-03-07T08:45:00Z,Bank A,IN,B001,M001,Alpha Mart,APAC
2,S003,640.0,settled,2026-03-07T09:15:00Z,Bank B,SG,B001,M001,Alpha Mart,APAC
3,S004,2100.0,settled,2026-03-07T08:20:00Z,Bank C,US,B002,M004,Delta Travels,US
4,S005,500.0,failed,2026-03-07T08:50:00Z,Bank C,US,B002,M004,Delta Travels,US


In [33]:
# Cleaning Column Names

api_normalize.columns = (
    api_normalize.columns
    .str.replace(".", "_", regex = False)
    .str.lower()
)

api_normalize.columns

Index(['settlement_id', 'amount_usd', 'status', 'processed_at', 'bank_name',
       'bank_country', 'batch_id', 'merchant_merchant_id',
       'merchant_merchant_name', 'merchant_region'],
      dtype='object')

In [34]:
# Date & Time Column Conversion

api_normalize['processed_at'] = pd.to_datetime(
    api_normalize['processed_at']
)

api_normalize.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 10 columns):
 #   Column                  Non-Null Count  Dtype              
---  ------                  --------------  -----              
 0   settlement_id           6 non-null      object             
 1   amount_usd              6 non-null      float64            
 2   status                  6 non-null      object             
 3   processed_at            6 non-null      datetime64[ns, UTC]
 4   bank_name               6 non-null      object             
 5   bank_country            6 non-null      object             
 6   batch_id                6 non-null      object             
 7   merchant_merchant_id    6 non-null      object             
 8   merchant_merchant_name  6 non-null      object             
 9   merchant_region         6 non-null      object             
dtypes: datetime64[ns, UTC](1), float64(1), object(8)
memory usage: 612.0+ bytes


In [35]:
# Manually Adding Generated_At Column to Normalized Dataset

api_normalize["generated_at"] = pd.to_datetime(
    api_data["generated_at"]
)

api_normalize.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 11 columns):
 #   Column                  Non-Null Count  Dtype              
---  ------                  --------------  -----              
 0   settlement_id           6 non-null      object             
 1   amount_usd              6 non-null      float64            
 2   status                  6 non-null      object             
 3   processed_at            6 non-null      datetime64[ns, UTC]
 4   bank_name               6 non-null      object             
 5   bank_country            6 non-null      object             
 6   batch_id                6 non-null      object             
 7   merchant_merchant_id    6 non-null      object             
 8   merchant_merchant_name  6 non-null      object             
 9   merchant_region         6 non-null      object             
 10  generated_at            6 non-null      datetime64[ns, UTC]
dtypes: datetime64[ns, UTC](2), float64(1), object(8)


In [36]:
api_normalize.head()

,settlement_id,amount_usd,status,processed_at,bank_name,bank_country,batch_id,merchant_merchant_id,merchant_merchant_name,merchant_region,generated_at
0,S001,1520.5,settled,2026-03-07 08:10:00+00:00,Bank A,IN,B001,M001,Alpha Mart,APAC,2026-03-07 10:00:00+00:00
1,S002,980.0,pending,2026-03-07 08:45:00+00:00,Bank A,IN,B001,M001,Alpha Mart,APAC,2026-03-07 10:00:00+00:00
2,S003,640.0,settled,2026-03-07 09:15:00+00:00,Bank B,SG,B001,M001,Alpha Mart,APAC,2026-03-07 10:00:00+00:00
3,S004,2100.0,settled,2026-03-07 08:20:00+00:00,Bank C,US,B002,M004,Delta Travels,US,2026-03-07 10:00:00+00:00
4,S005,500.0,failed,2026-03-07 08:50:00+00:00,Bank C,US,B002,M004,Delta Travels,US,2026-03-07 10:00:00+00:00


In [ ]:
# Exporting the Normalized API Dataset

api_normalize.to_csv(
    "../01_data/processed/api_normalized.csv",
    index=False
)

print("api_normalized.csv exported successfully")

api_normalized.csv exported successfully
